In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path

def season_code(start_year):
    return f"{start_year % 100:02d}{(start_year + 1) % 100:02d}"   # 2023 → "2324"

def load_pl(first=2010, last=2024, cache="data/pl_matches.csv"):
    path = Path(cache)
    if path.exists():
        return pd.read_csv(path, parse_dates=["Date"], low_memory=False)
    frames = []
    for y in range(first, last + 1):
        url = f"https://www.football-data.co.uk/mmz4281/{season_code(y)}/E0.csv"
        s = pd.read_csv(url, encoding="latin-1")      # older files aren't always UTF-8
        s = s.dropna(subset=["HomeTeam"])             # some files have blank trailing rows
        s["Season"] = f"{y}-{str(y + 1)[-2:]}"         # "2023-24"
        frames.append(s)
    out = pd.concat(frames, ignore_index=True)
    out["Date"] = pd.to_datetime(out["Date"], dayfirst=True, format="mixed")   # old seasons use dd/mm/yy
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, index=False)
    return out

pl = load_pl()
pl.groupby("Season").size()     # should be 380 for every season

,0
Season,
2010-11,380
2011-12,380
2012-13,380
2013-14,380
2014-15,380
2015-16,380
2016-17,380
2017-18,380
2018-19,380


In [ ]:
pl.isna().sum().sort_values(ascending=False).head()

,0
1XBA,5329
1XBD,5329
1XBH,5329
BFE>2.5,5323
BFE<2.5,5323


In [ ]:
pl["B365H"].isna().mean()

np.float64(0.0)

In [ ]:
pl.dropna(subset=["B365H", "B365D", "B365A"])        # drop rows missing odds
pl["HS"].fillna(pl["HS"].median())                   # fill with a typical value

,HS
0,23.0
1,7.0
2,13.0
3,18.0
4,6.0
...,...
5695,17.0
5696,10.0
5697,7.0
5698,4.0


### BUILD IT

In [ ]:
pl

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA
0,E0,2010-08-14,Aston Villa,West Ham,3.0,0.0,H,2.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,2010-08-14,Blackburn,Everton,1.0,0.0,H,1.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,2010-08-14,Bolton,Fulham,0.0,0.0,D,0.0,0.0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,2010-08-14,Chelsea,West Brom,6.0,0.0,H,2.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,2010-08-14,Sunderland,Birmingham,2.0,2.0,D,1.0,0.0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5695,NaN,2025-05-25,Newcastle,Everton,0.0,1.0,A,0.0,0.0,D,...,1.30,6.11,10.20,1.27,7.0,13.00,1.50,2.94,1.93,2.05
5696,NaN,2025-05-25,Nott'm Forest,Chelsea,0.0,1.0,A,0.0,0.0,D,...,3.50,3.86,2.06,3.55,3.9,2.14,1.76,2.28,1.86,2.14
5697,NaN,2025-05-25,Southampton,Arsenal,1.0,2.0,A,0.0,1.0,A,...,12.20,6.11,1.28,13.00,6.8,1.28,1.45,3.15,2.06,1.89
5698,NaN,2025-05-25,Tottenham,Brighton,1.0,4.0,A,1.0,0.0,H,...,3.63,4.12,1.95,3.95,4.2,1.93,1.51,2.92,2.06,1.93


In [ ]:
pl24 = pl.loc[pl["Season"] == "2023-24"]
pl24org = pl24.copy()
df = pl24.copy()

In [ ]:
pl24.head()
pl24 = pl24.copy()
pl24 = pl24.reset_index()

In [ ]:
pl24.tail()

,index,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA
375,5315,E0,2024-05-19,Crystal Palace,Aston Villa,5.0,0.0,H,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
376,5316,E0,2024-05-19,Liverpool,Wolves,2.0,0.0,H,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
377,5317,E0,2024-05-19,Luton,Fulham,2.0,4.0,A,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
378,5318,E0,2024-05-19,Man City,West Ham,3.0,1.0,H,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
379,5319,E0,2024-05-19,Sheffield United,Tottenham,0.0,3.0,A,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
home = pd.DataFrame({
    "Date":pl24["Date"],
    "Team":pl24["HomeTeam"],
    "Opponent":pl24["AwayTeam"],
    "GS":pl24["FTHG"],
    "GC":pl24["FTAG"],
    "GD":pl24["FTHG"]-pl24["FTAG"],
})
home

,Date,Team,Opponent,GS,GC,GD
0,2023-08-11,Burnley,Man City,0.0,3.0,-3.0
1,2023-08-12,Arsenal,Nott'm Forest,2.0,1.0,1.0
2,2023-08-12,Bournemouth,West Ham,1.0,1.0,0.0
3,2023-08-12,Brighton,Luton,4.0,1.0,3.0
4,2023-08-12,Everton,Fulham,0.0,1.0,-1.0
...,...,...,...,...,...,...
375,2024-05-19,Crystal Palace,Aston Villa,5.0,0.0,5.0
376,2024-05-19,Liverpool,Wolves,2.0,0.0,2.0
377,2024-05-19,Luton,Fulham,2.0,4.0,-2.0
378,2024-05-19,Man City,West Ham,3.0,1.0,2.0


In [ ]:
away = pd.DataFrame({
    "Date":pl24["Date"],
    "Team":pl24["AwayTeam"],
    "Opponent":pl24["HomeTeam"],
    "GS":pl24["FTAG"],
    "GC":pl24["FTHG"],
    "GD":pl24["FTAG"]-pl24["FTHG"],
})
away

,Date,Team,Opponent,GS,GC,GD
0,2023-08-11,Man City,Burnley,3.0,0.0,3.0
1,2023-08-12,Nott'm Forest,Arsenal,1.0,2.0,-1.0
2,2023-08-12,West Ham,Bournemouth,1.0,1.0,0.0
3,2023-08-12,Luton,Brighton,1.0,4.0,-3.0
4,2023-08-12,Fulham,Everton,1.0,0.0,1.0
...,...,...,...,...,...,...
375,2024-05-19,Aston Villa,Crystal Palace,0.0,5.0,-5.0
376,2024-05-19,Wolves,Liverpool,0.0,2.0,-2.0
377,2024-05-19,Fulham,Luton,4.0,2.0,2.0
378,2024-05-19,West Ham,Man City,1.0,3.0,-2.0


In [ ]:
data = pd.concat([home,away], ignore_index=True)
data = data.copy()

In [ ]:
tm = pd.concat([home, away], ignore_index=True).sort_values(["Team", "Date"])

In [ ]:
tm["Points"] = np.select([tm["GD"]>0,tm["GD"]==0,tm["GD"]<0], [3,1,0])
tm
ex4 = tm.copy()

In [ ]:
tm.groupby("Team")["Points"].sum().sort_values(ascending=False)

,Points
Team,
Man City,91
Arsenal,89
Liverpool,82
Aston Villa,68
Tottenham,66
Chelsea,63
Man United,60
Newcastle,60
West Ham,52


In [ ]:
tm["W"] = (tm.GS > tm.GC).astype(int)
tm["D"] = (tm.GS == tm.GC).astype(int)
tm["L"] = (tm.GS < tm.GC).astype(int)
tm

,Date,Team,Opponent,GS,GC,GD,Points,W,D,L
1,2023-08-12,Arsenal,Nott'm Forest,2.0,1.0,1.0,3,1,0,0
398,2023-08-21,Arsenal,Crystal Palace,1.0,0.0,1.0,3,1,0,0
21,2023-08-26,Arsenal,Fulham,2.0,2.0,0.0,1,0,1,0
38,2023-09-03,Arsenal,Man United,3.0,1.0,2.0,3,1,0,0
427,2023-09-17,Arsenal,Everton,1.0,0.0,1.0,3,1,0,0
...,...,...,...,...,...,...,...,...,...,...
331,2024-04-24,Wolves,Bournemouth,0.0,1.0,-1.0,0,0,0,1
340,2024-04-27,Wolves,Luton,2.0,1.0,1.0,3,1,0,0
732,2024-05-04,Wolves,Man City,1.0,5.0,-4.0,0,0,0,1
363,2024-05-11,Wolves,Crystal Palace,1.0,3.0,-2.0,0,0,0,1


In [ ]:
table = tm.groupby("Team").agg(
    P=("GS", "size"), W=("W", "sum"), D=("D", "sum"), L=("L", "sum"),
    GF=("GS", "sum"), GA=("GC", "sum"), GD = ("GD","sum"), Pts = ("Points","sum")
)
table.sort_values(["Pts","GD","GF"],ascending=False)

,P,W,D,L,GF,GA,GD,Pts
Team,,,,,,,,
Man City,38,28,7,3,96.0,34.0,62.0,91
Arsenal,38,28,5,5,91.0,29.0,62.0,89
Liverpool,38,24,10,4,86.0,41.0,45.0,82
Aston Villa,38,20,8,10,76.0,61.0,15.0,68
Tottenham,38,20,6,12,74.0,61.0,13.0,66
Chelsea,38,18,9,11,77.0,63.0,14.0,63
Newcastle,38,18,6,14,85.0,62.0,23.0,60
Man United,38,18,6,14,57.0,58.0,-1.0,60
West Ham,38,14,10,14,60.0,74.0,-14.0,52


In [ ]:
res = (pl.groupby("Season")["FTR"]
         .value_counts(normalize=True)
         .unstack()[["H", "D", "A"]]
         .round(3))
res

FTR,H,D,A
Season,,,
2010-11,0.471,0.292,0.237
2011-12,0.450,0.245,0.305
2012-13,0.437,0.284,0.279
2013-14,0.471,0.205,0.324
2014-15,0.453,0.245,0.303
2015-16,0.413,0.282,0.305
2016-17,0.492,0.221,0.287
2017-18,0.455,0.261,0.284
2018-19,0.476,0.187,0.337


In [ ]:
home = pd.DataFrame({
    "Team":pl24org["HomeTeam"],
    "Opponent":pl24org["AwayTeam"],
    "HTGD":pl24org["HTHG"]-pl24org["HTAG"],
    "FTGD":pl24org["FTHG"]-pl24org["FTAG"],
    "isHome":1,
})

away = pd.DataFrame({
    "Team":pl24org["AwayTeam"],
    "Opponent":pl24org["HomeTeam"],
    "HTGD":pl24org["HTAG"]-pl24org["HTHG"],
    "FTGD":pl24org["FTAG"]-pl24org["FTHG"],
    "isHome":0,
})

In [ ]:
result = pd.concat([home,away]).sort_values("Team")
result

,Team,Opponent,HTGD,FTGD,isHome
4987,Arsenal,Everton,0.0,1.0,0
5179,Arsenal,Burnley,2.0,5.0,0
5310,Arsenal,Everton,0.0,1.0,1
4994,Arsenal,Tottenham,0.0,0.0,1
5232,Arsenal,Man City,0.0,0.0,0
...,...,...,...,...,...
5292,Wolves,Man City,-3.0,-4.0,0
5021,Wolves,Bournemouth,-1.0,1.0,0
5107,Wolves,West Ham,-2.0,-3.0,0
4990,Wolves,Luton,0.0,0.0,0


In [ ]:
result["LosingFirstHalf"] = (result["HTGD"]<0).astype(int)

In [ ]:
result["Comeback"] = result["Comeback"] = (((result["HTGD"] < 0) & (result["FTGD"] >= 0))).astype(int)

In [ ]:
result

,Team,Opponent,HTGD,FTGD,isHome,LosingFirstHalf,Comeback
4987,Arsenal,Everton,0.0,1.0,0,0,0
5179,Arsenal,Burnley,2.0,5.0,0,0,0
5310,Arsenal,Everton,0.0,1.0,1,0,0
4994,Arsenal,Tottenham,0.0,0.0,1,0,0
5232,Arsenal,Man City,0.0,0.0,0,0,0
...,...,...,...,...,...,...,...
5292,Wolves,Man City,-3.0,-4.0,0,1,0
5021,Wolves,Bournemouth,-1.0,1.0,0,1,1
5107,Wolves,West Ham,-2.0,-3.0,0,1,0
4990,Wolves,Luton,0.0,0.0,0,0,0


In [ ]:
final = result.groupby("Team").agg(
  LosingFirstHalf = ("LosingFirstHalf", "sum"),
  Comeback = ("Comeback", "sum")
)

final["rate"] = final["Comeback"] / final["LosingFirstHalf"]

final[final.LosingFirstHalf>=5].sort_values("rate",ascending=False)

,LosingFirstHalf,Comeback,rate
Team,,,
Man City,7,6,0.857143
Liverpool,8,6,0.750000
Man United,11,5,0.454545
Crystal Palace,7,3,0.428571
Tottenham,12,5,0.416667
Chelsea,5,2,0.400000
West Ham,14,5,0.357143
Aston Villa,10,3,0.300000
Everton,10,3,0.300000


In [ ]:
df = df.copy()
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df["MatchID"] = range(len(df))

home = pd.DataFrame({"MatchID": df.MatchID, "Date": df.Date, "Team": df.HomeTeam, "GF": df.FTHG, "GA": df.FTAG})
away = pd.DataFrame({"MatchID": df.MatchID, "Date": df.Date, "Team": df.AwayTeam, "GF": df.FTAG, "GA": df.FTHG})
tm = pd.concat([home, away]).sort_values(["Team", "Date"])
tm["Pts"] = np.select([tm.GF > tm.GA, tm.GF == tm.GA], [3, 1], 0)
tm["Form5"] = tm.groupby("Team")["Pts"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())

form = tm[["MatchID", "Team", "Form5"]]
m = (df.merge(form.rename(columns={"Team": "HomeTeam", "Form5": "HomeForm"}), on=["MatchID", "HomeTeam"])
       .merge(form.rename(columns={"Team": "AwayTeam", "Form5": "AwayForm"}), on=["MatchID", "AwayTeam"]))
m = m.dropna(subset=["HomeForm", "AwayForm"])

better = m[m.HomeForm > m.AwayForm]
print("home win rate overall:          ", (m.FTR == "H").mean().round(3))
print("home win rate when better form: ", (better.FTR == "H").mean().round(3))

home win rate overall:           0.467
home win rate when better form:  0.6
